In [1]:
import sys
sys.path.append('/home/ronedr/evolution-strategy-baselines-comparison')

In [2]:
import jax
import jax.numpy as jnp
from evosax.problems import BBOBProblem, bbob_fns

In [3]:
seed = 0
key = jax.random.key(seed)

In [4]:
fn_name = "sphere"
num_dims = 10

problem = BBOBProblem(
    fn_name=fn_name,
    num_dims=num_dims,
    seed=seed,
)

problem_state = problem.init(key)

key, subkey = jax.random.split(key)
solution = problem.sample(subkey)

In [29]:
def metrics_fn(
        key,
        population,
        fitness,
        state,
        params,
):
    """Compute metrics for distribution-based algorithm."""
    best_idx_in_generation = jnp.argmin(fitness)
    return {
        "generation_counter": state.generation_counter,
        "best_fitness_in_generation": fitness[best_idx_in_generation],
        "best_solution_in_generation": population[best_idx_in_generation],
        "best_fitness": state.best_fitness,
        "best_solution": state.best_solution,
        "best_solution_norm": jnp.linalg.norm(state.best_solution),
        "mean_fitness_in_generation": jnp.mean(fitness)
        #jnp.nanmean(jnp.where(jnp.isinf(fitness), jnp.nan, fitness))
    }

In [30]:
from evosax.algorithms import LearnedES as ES

population_size = 256
es = ES(
    population_size=population_size,
    solution=solution,
    metrics_fn=metrics_fn
)

params = es.default_params

key, subkey = jax.random.split(key)
state = es.init(subkey, solution, params)

In [31]:
def step(carry, key):
    state, params, problem_state = carry
    key_ask, key_eval, key_tell = jax.random.split(key, 3)

    population, state = es.ask(key_ask, state, params)

    population = jnp.clip(population, -5, 5)
    fitness, problem_state, _ = problem.eval(key_eval, population, problem_state)

    state, metrics = es.tell(key_tell, population, fitness, state, params)
    
    return (state, params, problem_state), metrics

In [32]:
num_generations = 1000

key, subkey = jax.random.split(key)
keys = jax.random.split(subkey, num_generations)

_, metrics = jax.lax.scan(
    step,
    (state, params, problem_state),
    keys,
)

In [33]:
metrics

{'best_fitness': Array([164.59763 , 125.00185 ,  95.276566,  81.61191 ,  56.995564,
         53.092487,  47.49717 ,  46.225533,  46.225533,  45.798424,
         45.798424,  45.798424,  45.798424,  45.65614 ,  45.63526 ,
         45.63526 ,  45.63526 ,  45.63526 ,  45.63526 ,  45.63526 ,
         45.052536,  45.052536,  45.052536,  45.052536,  45.052536,
         45.052536,  44.799767,  44.799767,  44.799767,  44.741756,
         44.592594,  44.592594,  44.539192,  44.532887,  44.532887,
         44.49947 ,  44.48879 ,  44.4859  ,  44.462383,  44.462383,
         44.444984,  44.444984,  44.436413,  44.43494 ,  44.43494 ,
         44.42467 ,  44.42467 ,  44.422638,  44.41956 ,  44.41956 ,
         44.416775,  44.416775,  44.416775,  44.416775,  44.416775,
         44.416775,  44.416775,  44.416775,  44.416775,  44.416775,
         44.416775,  44.416775,  44.416775,  44.416775,  44.416775,
         44.416775,  44.416775,  44.416775,  44.416775,  44.416775,
         44.416775,  44.416775, 